# StormDesk triage model: fine-tune on a free GPU

Fine-tunes **Qwen3-1.7B** with a LoRA (Unsloth) on HumAID so it sorts disaster reports into 10
humanitarian categories, then measures macro F1 against the untuned model and saves the result.

**Runs on:** Google Colab or Kaggle with a free **T4** GPU (Colab: Runtime > Change runtime type > T4 GPU;
Kaggle: Settings > Accelerator > GPU T4). Expect roughly an hour end to end; this has not been timed.

**Before you start:** upload the three files made by `python training/prepare_humaid.py` on your
laptop (`training/data/train.jsonl`, `val.jsonl`, `test.jsonl`). Cell 2 asks for them on Colab;
on Kaggle, add them as a dataset.

Prompt format matches the StormDesk app exactly (Qwen3 chat template with thinking off), so the
model behaves the same when served by vLLM. Same settings as `training/train_triage.py`.

In [ ]:
# 1. GPU check and install (takes a few minutes)
!nvidia-smi -L
!pip install -q unsloth
!pip install -q scikit-learn

In [ ]:
# 2. Training data: train.jsonl, val.jsonl, test.jsonl from training/data on your laptop
import glob, os, shutil
DATA = "/content/data" if os.path.exists("/content") else "/kaggle/working/data"
os.makedirs(DATA, exist_ok=True)
for f in ("train.jsonl", "val.jsonl", "test.jsonl"):           # Kaggle: copy from an attached dataset
    for found in glob.glob(f"/kaggle/input/*/{f}") + glob.glob(f"/kaggle/input/*/*/{f}"):
        shutil.copy(found, f"{DATA}/{f}")
missing = [f for f in ("train.jsonl", "val.jsonl", "test.jsonl") if not os.path.exists(f"{DATA}/{f}")]
if missing:
    from google.colab import files                              # Colab: upload from your laptop
    print("Upload:", missing)
    for name, content in files.upload().items():
        open(f"{DATA}/{os.path.basename(name)}", "wb").write(content)
print({f: sum(1 for _ in open(f"{DATA}/{f}")) for f in ("train.jsonl", "val.jsonl", "test.jsonl")})

In [ ]:
# 3. Settings (same as training/train_triage.py)
BASE = "Qwen/Qwen3-1.7B"
EPOCHS = 1
LR = 2e-4
MAX_SEQ = 512
EVAL_N = 1000          # test examples scored for base and tuned (same sample); 15,160 available
QUICK_TEST = False     # True = 30 training steps only, to check everything runs before the real run
RESPONSES_ONLY = True  # learn only from the answer, not the system prompt and tweet (standard for classification)
PREVIOUS_BEST = 0.755  # macro F1 of the published model (v2); cell 8 only publishes a model that beats it
OUT = os.path.join(os.path.dirname(DATA), "stormdesk-triage")

# Must match stormdesk/triage.py CATEGORIES
CATEGORIES = [
    "requests_or_urgent_needs", "injured_or_dead_people", "missing_or_found_people",
    "displaced_people_and_evacuations", "infrastructure_and_utility_damage",
    "caution_and_advice", "rescue_volunteering_or_donation_effort",
    "other_relevant_information", "sympathy_and_support", "not_humanitarian",
    "dont_know_cant_judge",
]

In [ ]:
# 4. Load the base model and score it before training
import json, os, random
os.environ["UNSLOTH_RETURN_LOGITS"] = "1"   # scoring reads the logits directly
import torch
from unsloth import FastLanguageModel, is_bfloat16_supported
from sklearn.metrics import accuracy_score, classification_report, f1_score

model, tok = FastLanguageModel.from_pretrained(BASE, max_seq_length=MAX_SEQ, load_in_4bit=True)
tok.padding_side = "left"

test_rows = [json.loads(l) for l in open(f"{DATA}/test.jsonl")]
random.seed(0)
sample = random.sample(test_rows, min(EVAL_N, len(test_rows)))

# Constrained scoring, like the app: vLLM's structured output forces a valid category, so here the
# model picks among the 11 categories. After '{"category": "' each category starts with a different
# token, so the most likely of those 11 tokens is exactly what forced-format greedy decoding picks.
PREFIX = '{"category": "'
def prompt_of(r):
    return tok.apply_chat_template(r["messages"][:2], tokenize=False, add_generation_prompt=True,
                                   enable_thinking=False) + PREFIX
_p = prompt_of(sample[0]); _n = len(tok(_p, add_special_tokens=False).input_ids)
FIRST_IDS = [tok(_p + c, add_special_tokens=False).input_ids[_n] for c in CATEGORIES]
assert len(set(FIRST_IDS)) == len(CATEGORIES), "categories share a first token; scoring needs a rewrite"

def evaluate(model, rows, batch=16):
    FastLanguageModel.for_inference(model)
    tok.padding_side = "left"
    preds = []
    for i in range(0, len(rows), batch):
        enc = tok([prompt_of(r) for r in rows[i:i + batch]], return_tensors="pt", padding=True,
                  add_special_tokens=False).to("cuda")
        with torch.no_grad():
            last = model(**enc).logits[:, -1, :]
        preds += [CATEGORIES[j] for j in last[:, FIRST_IDS].argmax(-1).tolist()]
    gold = [r["label"] for r in rows]
    labels = [c for c in CATEGORIES if c in set(gold)]
    return {"macro_f1": f1_score(gold, preds, average="macro", labels=labels, zero_division=0),
            "accuracy": accuracy_score(gold, preds),
            "report": classification_report(gold, preds, labels=labels, zero_division=0)}

base = evaluate(model, sample)
# Fairer baseline: the same untuned model, told the category names in its prompt
LIST_SYS = ('Classify this disaster report into exactly one humanitarian category. Reply with JSON only, '
            'like {"category": "<name>"}. Categories:\n' + "\n".join(f"- {c}" for c in CATEGORIES))
listed = evaluate(model, [{**r, "messages": [{"role": "system", "content": LIST_SYS}, r["messages"][1]]}
                          for r in sample])
print(f"BASE                  macro F1 {base['macro_f1']:.3f}  accuracy {base['accuracy']:.3f}")
print(f"BASE + category list  macro F1 {listed['macro_f1']:.3f}  accuracy {listed['accuracy']:.3f}")

In [ ]:
# 5. LoRA fine-tune
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32, lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth")
FastLanguageModel.for_training(model)
tok.padding_side = "right"

def fmt(ex):  # thinking off: the same format the app serves with
    return {"text": tok.apply_chat_template(ex["messages"], tokenize=False, enable_thinking=False)}

data = load_dataset("json", data_files={"train": f"{DATA}/train.jsonl", "val": f"{DATA}/val.jsonl"})
data = data.map(fmt, remove_columns=["messages", "label"])

trainer = SFTTrainer(
    model=model, tokenizer=tok,
    train_dataset=data["train"], eval_dataset=data["val"].select(range(min(1000, len(data["val"])))),
    args=SFTConfig(
        output_dir="ckpt", dataset_text_field="text", max_seq_length=MAX_SEQ,
        per_device_train_batch_size=16, gradient_accumulation_steps=2,
        num_train_epochs=EPOCHS, max_steps=30 if QUICK_TEST else -1, learning_rate=LR,
        warmup_ratio=0.03, lr_scheduler_type="cosine", logging_steps=20,
        eval_strategy="steps", eval_steps=200, save_strategy="no",
        bf16=is_bfloat16_supported(), fp16=not is_bfloat16_supported(), report_to="none"),
)
if RESPONSES_ONLY:
    # Loss only on the assistant's answer. Note: the logged loss is then much lower than a
    # full-sequence run, so don't compare loss numbers across the two settings; compare F1.
    from unsloth.chat_templates import train_on_responses_only
    trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n",
                                      response_part="<|im_start|>assistant\n")
stats = trainer.train()
print(f"trained in {stats.metrics['train_runtime'] / 60:.1f} min")

In [ ]:
# 6. Score the fine-tuned model on the same test sample
tuned = evaluate(model, sample)
print(f"BASE                  macro F1 {base['macro_f1']:.3f}  accuracy {base['accuracy']:.3f}")
print(f"BASE + category list  macro F1 {listed['macro_f1']:.3f}  accuracy {listed['accuracy']:.3f}")
print(f"TUNED                 macro F1 {tuned['macro_f1']:.3f}  accuracy {tuned['accuracy']:.3f}")
print(tuned["report"])

In [ ]:
# 7. Save the merged 16-bit model (what vLLM serves) and the numbers for the model card
model.save_pretrained_merged(OUT, tok, save_method="merged_16bit")
metrics = {"eval_n": len(sample), "base_macro_f1": base["macro_f1"], "base_accuracy": base["accuracy"],
           "base_listed_macro_f1": listed["macro_f1"], "base_listed_accuracy": listed["accuracy"],
           "tuned_macro_f1": tuned["macro_f1"], "tuned_accuracy": tuned["accuracy"],
           "train_minutes": stats.metrics["train_runtime"] / 60, "quick_test": QUICK_TEST,
           "train_examples": len(data["train"]), "responses_only": RESPONSES_ONLY}
json.dump(metrics, open(f"{OUT}/stormdesk_metrics.json", "w"), indent=2)
open(f"{OUT}/per_class_report.txt", "w").write(tuned["report"])
print(json.dumps(metrics, indent=2))

**Keep the model.** Pick one:
- **Hugging Face (recommended):** run cell 8. It asks you to log in and publishes to `<your username>/stormdesk-triage`.
  It keeps the existing model card; update its numbers if they changed (source: `training/MODEL_CARD.md`).
- **Download:** run cell 9 for a zip (about 3.5 GB).

In [ ]:
# 8. Publish to Hugging Face (asks you to log in)
assert not QUICK_TEST, "This is a 30-step test model; set QUICK_TEST = False and retrain before publishing"
assert tuned["macro_f1"] > PREVIOUS_BEST, (f"Macro F1 {tuned['macro_f1']:.3f} does not beat the published "
                                           f"{PREVIOUS_BEST}; keeping the published model")
from huggingface_hub import hf_hub_download, login, upload_file, whoami
login()
REPO = f"{whoami()['name']}/stormdesk-triage"   # your account
# push_to_hub_merged replaces README.md with Unsloth's auto card (wrong license, no description),
# so save the current hand-written card first and put it back afterwards.
try:
    card = open(hf_hub_download(REPO, "README.md", force_download=True), "rb").read()
except Exception:
    card = None   # first publish: no card yet; paste training/MODEL_CARD.md into the page afterwards
model.push_to_hub_merged(REPO, tok, save_method="merged_16bit")
if card:
    upload_file(path_or_fileobj=card, path_in_repo="README.md", repo_id=REPO,
                commit_message="Restore the StormDesk model card")
print(f"https://huggingface.co/{REPO}  (update the numbers in the model card if they changed)")

In [ ]:
# 9. Or download a zip of the merged model
shutil.make_archive(OUT, "zip", OUT)
try:
    from google.colab import files
    files.download(f"{OUT}.zip")
except ImportError:
    print(f"Kaggle: find {OUT}.zip in the Output panel")